<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Custom_Attention_SMS_Student_W6_J1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification

Welcome to the guided notebook for the *Custom Attention Mechanism & SMS Spam* daily challenge. Cells tagged as **PREFILLED** are ready to run as-is. Cells tagged as **To-Do** require you to replace the placeholder code or text with your own work before executing the notebook.


## Why are we doing this?
Modern NLP systems rely on attention. By rolling your own attention block and contrasting it with a pre-trained GPT-2 classifier, you will demystify how query/key/value flows shape downstream predictions on a real SMS spam dataset.

![Image](https://github.com/user-attachments/assets/bc4d5315-983b-4fc1-9011-25fa743bb25f)


## Learning objectives
- Implement a custom scaled dot-product attention layer from scratch.
- Explain the respective roles of queries, keys, and values.
- Fine-tune GPT-2 for binary spam classification and compare it to a custom model.
- Evaluate both systems with accuracy, precision, recall, and F1.
- Reflect on trade-offs between transformer-based and lightweight attention models.


> **Learning point**
> Work through each part sequentially. Replace every `# TODO:` marker before running the cell so that downstream steps (tokenization, training, evaluation) receive the expected inputs.


# Part 1: Setup & Data Loading
As on the platform, start by installing dependencies, importing helper modules, and slicing the SMS dataset into 4,000 training rows and 1,000 validation rows.


**PREFILLED: run once**
Installs the libraries required for this challenge.


In [ ]:
%pip install --quiet datasets evaluate transformers[sentencepiece]


Note: you may need to restart the kernel to use updated packages.


**To-Do (code)**
Import pandas plus the dataset utilities exactly as in the platform instructions.


In [ ]:
import pandas as pd
from datasets import Dataset
from datasets import load_dataset

**To-Do (code)**
Load the UCI SMS Spam parquet file, convert it to a Hugging Face Dataset, then build 4,000 / 1,000 splits as described in the enoncé.


In [ ]:
DATA_PATH = 'hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet'
df = pd.read_parquet(DATA_PATH)
hf_dataset = Dataset.from_pandas(df)

TRAIN_START = 0
TRAIN_END = 4000
VAL_START = 4000
VAL_END = 5000

train_ds = hf_dataset.select(range(TRAIN_START, TRAIN_END))
val_ds = hf_dataset.select(range(VAL_START, VAL_END))
display(df.head())

# Part 2: Tokenization Setup
Initialize the GPT-2 tokenizer, set a padding token, and prepare batched tokenization for both splits.


> **Learning point**
> GPT-2 does not define a pad token. Reusing the EOS token keeps inputs aligned with how the model was pretrained.


In [ ]:
from transformers import GPT2Tokenizer

MODEL_NAME = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
TEXT_COLUMN = 'sms'
PADDING_STRATEGY = 'max_length'
TRUNCATION_FLAG = True
MAX_SEQ_LEN = 64

def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        padding=PADDING_STRATEGY,
        truncation=TRUNCATION_FLAG,
        max_length=MAX_SEQ_LEN,
    )

train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

# Part 3: Pre-trained GPT-2 Classifier
Load GPT-2 with a classification head suited for binary spam detection.


In [ ]:
import torch
from transformers import GPT2ForSequenceClassification

NUM_LABELS = 2
model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    pad_token_id=tokenizer.eos_token_id,
)

# Part 4: Custom Attention Implementation
Build the simple attention layer, classifier, and data pipeline for the scratch model.


> **Learning point**
> Scaling the dot products by $1/\sqrt{d_k}$ keeps gradients stable and prevents the softmax from collapsing when embeddings grow. This opeeration is crucial for training deep attention models.

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, value), attn

class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn = Attention(embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embed = self.embedding(x)
        attn_output, _ = self.attn(embed, embed, embed)
        pooled = attn_output.mean(dim=1)
        return self.fc(pooled)

> **Learning point**
> Tokenize once and reuse the same 64-token cap so both models receive comparable context windows.


In [ ]:
ATTN_TEXT_COLUMN = 'sms'
ATTN_MAX_LEN = 64

def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example[ATTN_TEXT_COLUMN],
        max_length=ATTN_MAX_LEN,
        truncation=True,
        padding='max_length',
    )
    return {'input_ids': tokens, 'label': example['label']}

train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)

In [ ]:
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'label': torch.tensor(item['label'], dtype=torch.long),
        }

train_loader = DataLoader(SMSDataset(train_ds_attn), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(val_ds_attn), batch_size=32)

In [ ]:
vocab_size = len(tokenizer)
embed_dim = 64
num_classes = 2
learning_rate = 1e-3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer = torch.optim.Adam(attn_model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print('Custom Attention model trained on SMS dataset. Sample batch loss:', loss.item())

# Part 5: Metrics & Evaluation
Load accuracy, precision, recall, and F1 from `evaluate`, then implement the shared `compute_metrics` helper.


In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load('accuracy')
precision = evaluate.load('precision')
recall = evaluate.load('recall')
f1 = evaluate.load('f1')

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'precision': precision.compute(predictions=preds, references=labels)['precision'],
        'recall': recall.compute(predictions=preds, references=labels)['recall'],
        'f1': f1.compute(predictions=preds, references=labels)['f1'],
    }

> **Learning point**
> Use the same helper dictionary pattern for both GPT-2 and the custom model so you can compare metrics side by side.


In [ ]:
gpt2_preds = []
gpt2_labels = []
model.to(device)
model.eval()
for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])

gpt2_metrics = {
    'accuracy': accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)['accuracy'],
    'precision': precision.compute(predictions=gpt2_preds, references=gpt2_labels)['precision'],
    'recall': recall.compute(predictions=gpt2_preds, references=gpt2_labels)['recall'],
    'f1': f1.compute(predictions=gpt2_preds, references=gpt2_labels)['f1'],
}
print('GPT-2 Metrics:', gpt2_metrics)

In [ ]:
attn_preds = []
attn_labels = []
attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())

attn_metrics = {
    'accuracy': accuracy.compute(predictions=attn_preds, references=attn_labels)['accuracy'],
    'precision': precision.compute(predictions=attn_preds, references=attn_labels)['precision'],
    'recall': recall.compute(predictions=attn_preds, references=attn_labels)['recall'],
    'f1': f1.compute(predictions=attn_preds, references=attn_labels)['f1'],
}
print('Attention Model Metrics:', attn_metrics)

# Part 6: Reflection Questions
Answer directly in the markdown cells below once your experiments finish.


### 1. Quels sont les rôles de la requête, de la clé et de la valeur dans le mécanisme d'attention ?

*   **Requête (Query) :** Elle représente l'élément actuel qui cherche des informations ou du contexte auprès des autres mots de la séquence.
*   **Clé (Key) :** Elle sert d'étiquette ou d'identifiant pour chaque mot, permettant de calculer un score de pertinence par rapport à la requête.
*   **Valeur (Value) :** Elle contient l'information réelle du mot qui sera pondérée par le score d'attention pour construire la nouvelle représentation contextuelle.

### 2. Pourquoi utilisons-nous un facteur d'échelle (par exemple, 1/√d_k) dans l'attention du produit scalaire ?

Nous utilisons ce facteur pour éviter que le produit scalaire ne devienne extrêmement grand lorsque la dimension des embeddings ($d_k$) augmente. Sans cela, les valeurs en entrée de la fonction softmax seraient très disparates, poussant les gradients vers des zones de saturation (presque nuls), ce qui rendrait l'entraînement du modèle très difficile ou instable.

### 3. En quoi l'auto-attention diffère-t-elle des modèles de séquences traditionnels comme les RNN ?

Contrairement aux RNN qui traitent les mots de manière séquentielle (un par un), l'auto-attention traite toute la séquence en parallèle, ce qui est beaucoup plus efficace sur GPU. De plus, l'attention permet de capturer des dépendances à longue distance instantanément, alors que les RNN souffrent souvent de l'oubli de l'information au fil du temps.

### 4. Analyse des performances

En général, le modèle **GPT-2** obtient de meilleurs résultats (F1-score plus élevé) car il bénéficie d'un pré-entraînement massif sur des milliards de mots, lui permettant de comprendre la sémantique complexe. Le modèle personnalisé, bien que rapide et léger, part de zéro et ne possède qu'une seule couche d'attention très simple.

**Avantages/Inconvénients :** GPT-2 est performant mais lourd et coûteux en calcul ; le modèle personnalisé est extrêmement rapide mais moins précis. Pour améliorer le modèle personnalisé, on pourrait ajouter plusieurs têtes d'attention (Multi-Head Attention) ou empiler plusieurs couches de Transformers.